# Notebook 1 — Ingesta batch: API pública → BigQuery
### Serie *Capa 1 del embudo AI-first · Ingesta de datos en GCP*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelserdna/capa1-ingesta-colab/blob/main/01_batch_api_a_bigquery.ipynb)

---

## Qué vas a conseguir aquí

Vas a construir tu **primera ingesta de datos real**: traer información de una fuente externa y dejarla en BigQuery, lista para que más adelante una IA razone sobre ella. Al terminar sabrás:

1. Qué es la **ingesta batch** (por lotes) y cuándo es el modo adecuado.
2. Llamar a una **API pública** (clima, sin clave) y entender su respuesta JSON.
3. Qué es la **paginación** y cómo recorrer datos que vienen "de página en página".
4. **Transformar** datos crudos a un **esquema canónico** limpio con pandas.
5. Escribir en BigQuery con **MERGE** para que la ingesta sea **idempotente** (ejecutarla dos veces no duplica).
6. Hacer ingesta **incremental** (traer solo lo nuevo).
7. Desplegar todo esto en producción con un **prompt para Claude Code + gcloud**.

**Requisito:** haber completado el **Notebook 0** (entorno en verde). **Tiempo:** ~60 min.


## Mapa mental — dónde estamos

```
   FUENTE            INGESTA (Capa 1)          ALMACÉN (Capa 2)
  ┌────────┐   batch periódico (este NB)     ┌──────────────┐
  │ API    │ ─────────────────────────────▶ │  BigQuery    │ ──▶ IA razona
  │ clima  │   pedir · transformar · MERGE   │  (raw)       │
  └────────┘                                 └──────────────┘
```

De los **4 modos de ingesta** (Notebook 0), aquí trabajamos el primero: **batch periódico** — datos que no cambian al segundo y que traemos en lotes cada cierto tiempo (cada noche, cada hora). Es el modo más común y por donde empieza casi toda empresa.


## Setup de la sesión

Igual que en el Notebook 0: autenticarte, fijar tu proyecto e instalar librerías. **Cambia `TU_PROJECT_ID`** por el tuyo.


In [ ]:
# 1) Autenticación con tu cuenta de Google
from google.colab import auth
auth.authenticate_user()

# 2) Variables del proyecto  (CAMBIA TU_PROJECT_ID)
PROJECT_ID = 'TU_PROJECT_ID'
REGION = 'europe-west1'
DATASET = 'capa1_trabajo'          # creado en el Notebook 0
TABLA = 'raw_clima_observaciones'  # la crearemos aquí

import os
os.environ['PROJECT_ID'] = PROJECT_ID
!gcloud config set project $PROJECT_ID -q

# 3) Librerías (db-dtypes permite pasar resultados de BigQuery a pandas)
!pip install -q google-cloud-bigquery db-dtypes
print('Setup listo.')


## 1.1 · ¿Qué es la ingesta batch?

**Batch** = "por lotes". En vez de procesar cada dato en el instante en que ocurre, **acumulas** y traes un grupo entero cada cierto tiempo.

- **Cuándo usarlo:** datos que no cambian al segundo — un CRM, una tabla de facturación, el clima del día, un catálogo de productos. Si que el dato llegue "esta noche" en lugar de "ahora mismo" es aceptable, batch es lo correcto (y lo más barato y simple).
- **Cuándo NO:** si necesitas reaccionar al instante (un pago, un clic, un sensor) → eso es *streaming* o *event-driven* (Notebooks 3 y 4).

Una ingesta batch siempre tiene las mismas **3 fases**:

1. **Extraer** — pedir los datos a la fuente (una API, una BD, un archivo).
2. **Transformar** — limpiarlos y darles la forma que queremos (el *esquema canónico*).
3. **Cargar** — escribirlos en el destino (BigQuery), de forma que repetir la carga no rompa nada (*idempotencia*).

Esto es el patrón **ETL** (Extract, Transform, Load). Vamos fase por fase.


## 1.2 · Extraer — llamar a la API de clima

Usamos **Open-Meteo**: una API pública y **gratuita, sin clave**, que da el pronóstico del tiempo. Le pedimos, para una ciudad, las temperaturas máximas/mínimas y la lluvia de los próximos días.

Fíjate en la estructura de la respuesta: dentro de `daily` hay **listas paralelas** — una de fechas, una de temperaturas máximas, etc. La posición `i` de cada lista es el mismo día.


In [ ]:
import requests

def pedir_clima(lat, lon):
    """Pide a Open-Meteo el clima diario de una coordenada. Devuelve el JSON (dict)."""
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat,
        'longitude': lon,
        'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum',
        'timezone': 'Europe/Madrid',
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()   # si la API devuelve error (4xx/5xx), lanza excepción
    return r.json()

# Probar con Madrid
ejemplo = pedir_clima(40.4168, -3.7038)
print('Claves de la respuesta:', list(ejemplo.keys()))
print('Primeros 3 días:', ejemplo['daily']['time'][:3])
print('Temp. máximas:', ejemplo['daily']['temperature_2m_max'][:3])


## 1.3 · Paginación — cuando los datos vienen "de página en página"

Muchas APIs **no te dan todo de golpe**: si hay 10.000 registros, te los entregan en **páginas** (100 cada vez) y tú vas pidiendo página 1, 2, 3… hasta que no quedan. Saberlo es esencial: si solo lees la primera página, **pierdes datos sin darte cuenta**.

Open-Meteo nos devuelve todo de una vez (no pagina), así que para *ver* el patrón usamos una API de práctica, **JSONPlaceholder**, que sí pagina con `_page` y `_limit`.

El patrón universal: **un bucle que pide páginas hasta que una vuelve vacía.**


In [ ]:
def traer_todo_paginado(url, limit=20):
    """Recorre todas las páginas de una API hasta que una venga vacía."""
    resultados = []
    page = 1
    while True:
        r = requests.get(url, params={'_page': page, '_limit': limit}, timeout=30)
        lote = r.json()
        if not lote:                # lista vacía -> no hay más páginas -> salimos
            break
        resultados.extend(lote)
        print(f'  página {page}: {len(lote)} registros (acumulado {len(resultados)})')
        page += 1
    return resultados

posts = traer_todo_paginado('https://jsonplaceholder.typicode.com/posts')
print('TOTAL recorriendo todas las páginas:', len(posts))


> **Recuerda este bucle.** En el Notebook 1 de la clase original, la API de Notion paginaba con un campo `next_cursor`; el principio es idéntico: *seguir pidiendo hasta que no haya más*. Aquí Open-Meteo no lo necesita, pero en una ingesta real lo tendrás casi siempre.


## 1.4 · Transformar — del JSON crudo al esquema canónico

Los datos crudos de la API tienen **su** forma (listas paralelas dentro de `daily`). Nosotros queremos **nuestra** forma: una tabla limpia con una fila por (ciudad, día) y columnas claras. A eso lo llamamos **esquema canónico** — el formato estándar que toda la empresa entiende.

Esquema objetivo:

| columna | tipo | significado |
|---|---|---|
| `ciudad` | STRING | nombre de la ciudad |
| `fecha` | DATE | día de la observación |
| `temp_max` | FLOAT | temperatura máxima (°C) |
| `temp_min` | FLOAT | temperatura mínima (°C) |
| `precip_mm` | FLOAT | precipitación (mm) |

Vamos a pedir varias ciudades y **aplanar** las listas paralelas en filas.


In [ ]:
import pandas as pd

CIUDADES = {
    'Madrid':    (40.4168, -3.7038),
    'Barcelona': (41.3874,  2.1686),
    'Sevilla':   (37.3891, -5.9845),
}

filas = []
for ciudad, (lat, lon) in CIUDADES.items():
    d = pedir_clima(lat, lon)['daily']
    # recorrer las listas paralelas: la posición i es el mismo día en todas
    for i in range(len(d['time'])):
        filas.append({
            'ciudad': ciudad,
            'fecha': d['time'][i],
            'temp_max': d['temperature_2m_max'][i],
            'temp_min': d['temperature_2m_min'][i],
            'precip_mm': d['precipitation_sum'][i],
        })

df = pd.DataFrame(filas)
df['fecha'] = pd.to_datetime(df['fecha']).dt.date   # convertir texto -> fecha real
print(f'{len(df)} filas ({df.ciudad.nunique()} ciudades)')
df.head(8)


## 1.5 · Cargar — escribir en BigQuery con MERGE (idempotencia)

Aquí está la lección más importante de toda la ingesta: la **idempotencia**.

> **Idempotente** = ejecutar la operación una vez o diez veces deja el mismo resultado.

¿Por qué importa? Porque las ingestas **se reintentan**: una red falla, un proceso se reinicia, alguien lanza el job dos veces. Si cargas con `INSERT` (añadir), cada reintento **duplica** los datos. Si cargas con `MERGE` (combinar por una clave), un reintento simplemente **actualiza** la fila existente. Cero duplicados.

Nuestra **clave natural** es `(ciudad, fecha)`: no puede haber dos observaciones del mismo día y ciudad.

El patrón profesional tiene 3 pasos:
1. Cargar el lote nuevo en una tabla **staging** (temporal).
2. Hacer `MERGE` de staging hacia la tabla **destino** por la clave.
3. (BigQuery limpia la staging sola, o la reusamos.)


In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

destino  = f'{PROJECT_ID}.{DATASET}.{TABLA}'
staging  = f'{PROJECT_ID}.{DATASET}.{TABLA}_staging'

# Crear la tabla destino si no existe, con su esquema explícito
schema = [
    bigquery.SchemaField('ciudad', 'STRING'),
    bigquery.SchemaField('fecha', 'DATE'),
    bigquery.SchemaField('temp_max', 'FLOAT'),
    bigquery.SchemaField('temp_min', 'FLOAT'),
    bigquery.SchemaField('precip_mm', 'FLOAT'),
]
client.create_table(bigquery.Table(destino, schema=schema), exists_ok=True)
print('Tabla destino lista:', destino)


In [ ]:
def cargar_con_merge(df):
    """Carga el DataFrame en staging y hace MERGE por (ciudad, fecha) hacia el destino."""
    # Paso 1: subir el lote a la tabla staging (se reemplaza cada vez)
    job_cfg = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition='WRITE_TRUNCATE',  # staging siempre limpia
    )
    client.load_table_from_dataframe(df, staging, job_config=job_cfg).result()

    # Paso 2: MERGE staging -> destino por la clave natural
    merge_sql = f'''
    MERGE `{destino}` T
    USING `{staging}` S
    ON T.ciudad = S.ciudad AND T.fecha = S.fecha
    WHEN MATCHED THEN UPDATE SET
      temp_max = S.temp_max, temp_min = S.temp_min, precip_mm = S.precip_mm
    WHEN NOT MATCHED THEN INSERT (ciudad, fecha, temp_max, temp_min, precip_mm)
      VALUES (S.ciudad, S.fecha, S.temp_max, S.temp_min, S.precip_mm)
    '''
    client.query(merge_sql).result()

def contar():
    return list(client.query(f'SELECT COUNT(*) c FROM `{destino}`').result())[0].c

# Primera carga
cargar_con_merge(df)
print('Filas tras la 1ª carga:', contar())


### La prueba de la idempotencia

Ejecuta la **misma carga otra vez**. Si fuera `INSERT`, el contador se duplicaría. Con `MERGE`, se queda **igual**: ese es el objetivo.


In [ ]:
cargar_con_merge(df)   # exactamente la misma carga
print('Filas tras la 2ª carga (debe ser IGUAL):', contar())


In [ ]:
# Ver una muestra de lo que quedó en BigQuery
q = f'SELECT ciudad, fecha, temp_max, temp_min, precip_mm FROM `{destino}` ORDER BY ciudad, fecha LIMIT 10'
client.query(q).result().to_dataframe()


## 1.6 · Ingesta incremental — traer solo lo nuevo

En producción no quieres traer **todo** el histórico cada noche: es lento y caro. Quieres traer **solo lo que cambió** desde la última ejecución. El patrón:

1. Guardas un **marcador** (`last_run_at`) con la fecha/hora de la última carga.
2. En la siguiente ejecución, pides a la fuente solo lo que cambió **después** de ese marcador.
3. Al terminar, actualizas el marcador.

Open-Meteo no filtra por fecha de cambio, así que aquí lo mostramos como **concepto**: una pequeña tabla de control en BigQuery que guarda el marcador. En una fuente real (un CRM, una BD), pasarías ese marcador como filtro en la petición.


In [ ]:
# Tabla de control con el marcador de la última ejecución
control = f'{PROJECT_ID}.{DATASET}.ingesta_control'
client.query(f'''
  CREATE TABLE IF NOT EXISTS `{control}` (fuente STRING, last_run_at TIMESTAMP)
''').result()

# Leer el marcador actual (si no existe, es la primera vez)
prev = list(client.query(f"SELECT last_run_at FROM `{control}` WHERE fuente='clima'").result())
print('Última ejecución registrada:', prev[0].last_run_at if prev else 'nunca (primera vez)')

# ... aquí iría la petición filtrada por ese marcador ...

# Actualizar el marcador a 'ahora' con MERGE (idempotente también aquí)
client.query(f'''
  MERGE `{control}` T USING (SELECT 'clima' AS fuente, CURRENT_TIMESTAMP() AS ts) S
  ON T.fuente = S.fuente
  WHEN MATCHED THEN UPDATE SET last_run_at = S.ts
  WHEN NOT MATCHED THEN INSERT (fuente, last_run_at) VALUES (S.fuente, S.ts)
''').result()
print('Marcador actualizado a ahora.')


## 1.7 · Llevarlo a producción — prompt para Claude Code + gcloud

Lo que has hecho en este cuaderno corre **mientras el Colab está abierto**. En producción, una ingesta batch debe correr **sola cada noche**, sin nadie delante. La forma estándar en GCP: un **Cloud Run Job** (tu código empaquetado) disparado por **Cloud Scheduler** (un cron en la nube).

En vez de escribir todo eso a mano, pega este prompt en **Claude Code** (en una terminal con `gcloud` autenticado). Claude Code generará el código, el Dockerfile y los comandos, y los **desplegará**:

```
Eres un ingeniero senior de GCP con gcloud autenticado en el proyecto <TU_PROJECT_ID> (region europe-west1).

CONTEXTO
- Dataset destino: capa1_trabajo, tabla raw_clima_observaciones (ciudad STRING, fecha DATE,
  temp_max FLOAT, temp_min FLOAT, precip_mm FLOAT).
- Fuente: API pública Open-Meteo (https://api.open-meteo.com/v1/forecast), sin clave.
- Ciudades: Madrid, Barcelona, Sevilla (lat/lon conocidas).

OBJETIVO
1. Crea un Cloud Run Job en Python que: pida el clima diario de las 3 ciudades,
   transforme al esquema canónico y haga MERGE por (ciudad, fecha) en la tabla destino.
2. Crea una service account dedicada con permisos mínimos sobre ese dataset.
3. Programa el job con Cloud Scheduler todos los días a las 03:00 Europe/Madrid.

RESTRICCIONES
- Idempotencia (MERGE, nunca INSERT) · Retries con backoff en las llamadas HTTP.
- Sin secretos en claro · Logging a Cloud Logging · Nada de IDs hardcodeados (usa variables).

EJECUCIÓN
- Ejecuta los comandos gcloud tú mismo, uno a uno, mostrando la salida.
- Antes de cualquier comando destructivo, párate y pídeme confirmación.
- Al final, lanza el job una vez y dame la query de verificación.
```

> Esto es **AI-augmented engineering**: tú aportas la arquitectura (qué, dónde, con qué restricciones); la IA produce y despliega. Tu trabajo es **revisar** lo que genera (¿hay idempotencia? ¿retries? ¿secretos?) antes de aprobar.


## Trampas comunes (provócalas para entenderlas)

### Trampa 1 — Usar INSERT en vez de MERGE
La celda de abajo inserta el lote con `INSERT`. Ejecútala **dos veces** y mira cómo el contador **se duplica**: justo lo que el MERGE evita. (Luego limpiamos.)


In [ ]:
# ANTIPATRÓN a propósito: append directo (NO idempotente)
client.load_table_from_dataframe(
    df, destino,
    job_config=bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')
).result()
print('Filas tras INSERT directo:', contar(), '  <-- crece cada vez = duplicados')


In [ ]:
# Limpieza: volver a dejar la tabla correcta con MERGE desde cero
client.query(f'TRUNCATE TABLE `{destino}`').result()
cargar_con_merge(df)
print('Restaurado con MERGE. Filas correctas:', contar())


### Trampa 2 — Secretos en el código
Open-Meteo no necesita clave, pero la mayoría de APIs sí. **Nunca** escribas la clave en el código (`API_KEY = 'abc123'`): acaba en Git, en logs, compartido. Lo correcto es **Secret Manager**, que usarás desde el Notebook 2. Recuérdalo: *si una credencial aparece en una celda, está mal*.


## Reto (entrega)

Replica el patrón ETL con una **fuente distinta**: un **CSV público alojado en Cloud Storage**.

1. Descarga un CSV público a un DataFrame (por ejemplo, exporta una muestra del dataset público de taxis a CSV, o usa cualquier CSV público).
2. Define un esquema canónico para sus columnas.
3. Cárgalo en una tabla nueva `raw_<loquesea>` con **MERGE** por una clave natural.
4. Demuestra la idempotencia ejecutando dos veces.

Pista de extracción desde GCS:
```python
df = pd.read_csv('gs://nombre-bucket/ruta/archivo.csv', storage_options={'token': 'cloud'})
```


## Checklist de salida

- [ ] Llamaste a una API pública y entendiste su JSON.
- [ ] Recorriste páginas con un bucle hasta agotarlas.
- [ ] Transformaste datos crudos a un esquema canónico con pandas.
- [ ] Cargaste con **MERGE** y comprobaste que **2 cargas = mismo número de filas**.
- [ ] Entiendes por qué la idempotencia es obligatoria en ingesta.
- [ ] Sabes qué prompt darle a Claude Code para desplegarlo en producción.

**Siguiente:** `02_workflows.ipynb` — orquestar varios pasos con condiciones y paralelismo, desplegado de verdad con `gcloud workflows`.
